# Chapitre 2 — Graphes eulériens, hamiltoniens et planaires
**Support d'exécution**

**Installation si nécessaire :** exécuter la cellule suivante une seule fois.

In [ ]:
%pip install networkx matplotlib

In [5]:
import networkx as nx
import matplotlib.pyplot as plt

def dessiner_graphe(G, titre, pos=None, node_color="#bfdbfe", edge_color="#64748b"):
    if pos is None:
        pos = nx.spring_layout(G, seed=7)
    plt.figure(figsize=(7, 4.5))
    nx.draw_networkx(
        G, pos, with_labels=True, node_color=node_color,
        edge_color=edge_color, node_size=900, font_size=11,
        font_weight="bold", width=2
    )
    plt.title(titre)
    plt.axis("off")
    plt.show()
    return pos


# I. Graphes eulériens

## 1. Rappels
- **Chaîne eulérienne** : chaque arête est parcourue exactement une fois.
- **Cycle eulérien** : chaîne eulérienne fermée.
- **Graphe eulérien** : possède un cycle eulérien.
- **Graphe semi-eulérien** : possède une chaîne eulérienne.

Pour un graphe connexe, le théorème d'Euler indique :
- chaîne eulérienne si le nombre de sommets de degré impair vaut 0 ou 2 ;
- cycle eulérien si tous les sommets sont de degré pair.

In [ ]:
# Exemple 1 : graphe eulérien
G_euler = nx.Graph()
G_euler.add_edges_from([
    ("A", "B"), ("B", "C"), ("C", "D"), ("D", "A"),
    ("A", "C"), ("C", "E"), ("E", "A")
])

pos_euler = {"A": (0, 0), "B": (1, 1), "C": (2, 0),
             "D": (1, -1), "E": (3, 0)}
dessiner_graphe(G_euler, "Exemple de graphe eulérien", pos_euler)

degres = dict(G_euler.degree())
impairs = [s for s, d in degres.items() if d % 2 == 1]
print("Degrés :", degres)
print("Sommets de degré impair :", impairs)
print("Graphe connexe :", nx.is_connected(G_euler))
print("Cycle eulérien :", nx.is_eulerian(G_euler))


## 2. Exécution de l'algorithme eulérien

NetworkX implémente un parcours eulérien. La fonction ci-dessous affiche le parcours sous forme d'une suite de sommets. On vérifie ensuite que chaque arête est utilisée une fois.

In [ ]:
if nx.is_eulerian(G_euler):
    parcours = list(nx.eulerian_circuit(G_euler, source="A"))
    suite_sommets = [parcours[0][0]] + [v for u, v in parcours]
    print("Arêtes parcourues :", parcours)
    print("Cycle eulérien :", " -> ".join(suite_sommets))
    print("Nombre d'arêtes parcourues :", len(parcours))
    print("Nombre d'arêtes du graphe :", G_euler.number_of_edges())
else:
    print("Ce graphe ne possède pas de cycle eulérien.")


## 3. Exemple semi-eulérien

Un graphe connexe ayant exactement deux sommets de degré impair admet une chaîne eulérienne dont les extrémités sont ces deux sommets.

In [ ]:
G_semi = nx.Graph()
G_semi.add_edges_from([
    ("A", "B"), ("B", "C"), ("C", "D"), ("D", "A"), ("A", "C")
])
pos_semi = {"A": (0, 0), "B": (1, 1), "C": (2, 0), "D": (1, -1)}
dessiner_graphe(G_semi, "Exemple semi-eulérien", pos_semi)

impairs_semi = [s for s, d in G_semi.degree() if d % 2 == 1]
print("Degrés :", dict(G_semi.degree()))
print("Sommets impairs :", impairs_semi)
print("Semi-eulérien :", nx.is_semieulerian(G_semi))
if nx.has_eulerian_path(G_semi):
    chemin = list(nx.eulerian_path(G_semi))
    print("Chaîne eulérienne (arêtes) :", chemin)
    print("Extrémités :", chemin[0][0], "et", chemin[-1][1])


# II. Graphes hamiltoniens

## 1. Rappels
- **Chaîne hamiltonienne** : visite chaque sommet exactement une fois.
- **Cycle hamiltonien** : chaîne hamiltonienne fermée.
- **Graphe hamiltonien** : possède un cycle hamiltonien.

À la différence du cas eulérien, le cours précise qu'il n'existe pas de condition nécessaire et suffisante simple permettant de décider dans tous les cas si un graphe est hamiltonien.

Quelques propriétés du cours :
- un sommet de degré 1 empêche l'existence d'un cycle hamiltonien ;
- le graphe complet \(K_n\), pour \(n\geq 3\), est hamiltonien ;
- une condition suffisante : pour toute paire de sommets non adjacents x,y, si d(x)+d(y) ≥ n, alors le graphe est hamiltonien.

In [ ]:
G_ham = nx.Graph()
G_ham.add_edges_from([
    ("A", "B"), ("B", "C"), ("C", "D"), ("D", "E"), ("E", "A"),
    ("A", "C"), ("B", "D")
])
pos_ham = {"A": (0, 0), "B": (1, 1), "C": (2, 1),
           "D": (3, 0), "E": (1.5, -1)}
dessiner_graphe(G_ham, "Exemple contenant le cycle A-B-C-D-E-A", pos_ham)


## 2. Recherche d'un cycle hamiltonien par retour arrière

Le code suivant est une méthode pédagogique de recherche exhaustive avec élagage. Il fixe un sommet de départ, ajoute un sommet adjacent non encore visité, puis vérifie à la fin si le dernier sommet est relié au départ.

In [ ]:
def trouver_cycle_hamiltonien(G, depart=None):
    sommets = list(G.nodes)
    if not sommets:
        return None
    if depart is None:
        depart = sommets[0]
    chemin = [depart]
    visites = {depart}

    def backtrack():
        if len(chemin) == len(sommets):
            return chemin + [depart] if G.has_edge(chemin[-1], depart) else None

        dernier = chemin[-1]
        for voisin in G.neighbors(dernier):
            if voisin not in visites:
                visites.add(voisin)
                chemin.append(voisin)
                resultat = backtrack()
                if resultat is not None:
                    return resultat
                chemin.pop()
                visites.remove(voisin)
        return None

    return backtrack()

cycle = trouver_cycle_hamiltonien(G_ham, depart="A")
print("Cycle hamiltonien trouvé :", cycle)
if cycle:
    print(" -> ".join(map(str, cycle)))
else:
    print("Aucun cycle hamiltonien trouvé.")


# III. Graphes planaires

## 1. Définitions
Un graphe est **planaire** s'il peut être représenté dans le plan sans croisement d'arêtes. Une **face** est une région du plan délimitée par les arêtes ; la région extérieure compte aussi.

Pour un graphe planaire connexe :
- formule d'Euler : S − A + R = 2 ;
- somme des degrés des faces : somme d(f) = 2A ;
- pour un graphe simple planaire connexe : A ≤ 3S − 6 ;
- si le graphe est en plus sans triangle : A ≤ 2S − 4.

Le cours cite aussi le théorème de Kuratowski : K₅ et K₃,₃ sont non planaires.

In [ ]:
G_plan = nx.Graph()
G_plan.add_edges_from([(1,2), (2,3), (3,4), (4,5), (5,1), (1,3), (1,4)])
pos_plan = {1:(0,0), 2:(1,1), 3:(2,1), 4:(2,-1), 5:(1,-1)}
dessiner_graphe(G_plan, "Graphe planaire (dessin sans croisement)", pos_plan)

est_planaire, certificat = nx.check_planarity(G_plan)
print("Le graphe est planaire :", est_planaire)
if est_planaire:
    print("Représentation planaire calculée par NetworkX.")
    pos = nx.planar_layout(G_plan)
    dessiner_graphe(G_plan, "Disposition planaire calculée", pos)


## 2. Vérification de la formule d'Euler

Dans un graphe planaire connexe, on peut calculer le nombre de faces par R = 2 − S + A.

In [ ]:
S = G_plan.number_of_nodes()
A = G_plan.number_of_edges()
R = 2 - S + A
print(f"Nombre de sommets S = {S}")
print(f"Nombre d'arêtes A = {A}")
print(f"Nombre de faces R = 2 - {S} + {A} = {R}")
print("Vérification S - A + R =", S - A + R)


## 3. Comparaison avec K₅ et K₃,₃

Ces deux graphes sont des exemples classiques de graphes non planaires mentionnés dans le cours.

In [ ]:
K5 = nx.complete_graph(5)
K33 = nx.complete_bipartite_graph(3, 3)

for nom, G in [("K5", K5), ("K3,3", K33)]:
    planaire, _ = nx.check_planarity(G)
    print(f"{nom} : planaire ? {planaire}")


# IV. Coloration — algorithme de Welch-Powell


Colorier les sommets de sorte que deux sommets adjacents aient des couleurs différentes. Le nombre chromatique est le nombre minimal de couleurs.

**Welch-Powell :**
1. Trier les sommets par degré décroissant.
2. Donner une couleur au premier sommet non coloré.
3. Donner la même couleur aux sommets non colorés non adjacents aux sommets déjà choisis pour cette couleur.
4. Recommencer avec une nouvelle couleur jusqu'à coloration de tous les sommets.

L'algorithme est une heuristique : il produit une coloration valide, mais pas nécessairement optimale dans tous les graphes.

In [ ]:
def welch_powell(G):
    ordre = sorted(G.nodes, key=lambda s: G.degree(s), reverse=True)
    couleurs = {}
    couleur_id = 0

    for sommet in ordre:
        if sommet in couleurs:
            continue
        couleur_id += 1
        couleurs[sommet] = couleur_id
        for candidat in ordre:
            if candidat in couleurs:
                continue
            if all(not G.has_edge(candidat, s)
                   for s, c in couleurs.items() if c == couleur_id):
                couleurs[candidat] = couleur_id
    return couleurs, ordre

G_color = nx.Graph()
G_color.add_edges_from([
    (1,2),(1,3),(1,4),(2,3),(2,5),
    (3,4),(3,5),(4,5),(4,6),(5,6)
])
couleurs, ordre = welch_powell(G_color)
print("Ordre décroissant des degrés :", ordre)
print("Degrés :", dict(G_color.degree()))
print("Coloration obtenue :", couleurs)
print("Nombre de couleurs utilisées :", len(set(couleurs.values())))


In [ ]:
palette = ["#fca5a5", "#93c5fd", "#86efac", "#fde68a",
           "#c4b5fd", "#fdba74", "#a5f3fc"]
node_colors = [palette[couleurs[n] - 1] for n in G_color.nodes]
dessiner_graphe(G_color, "Coloration obtenue par Welch-Powell",
                node_color=node_colors)
